<a href="https://www.kaggle.com/code/taha524/ml-project-phase-3-rollbcsf23m524?scriptVersionId=305028997" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

#  *AI vs Human Text Detection* Feature Engineering


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style="darkgrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120
print("Libraries loaded ✅")

Libraries loaded ✅


Dataset Overview

load the data and analyze it

In [2]:
df = pd.read_csv('/kaggle/input/datasets/algozee/ai-generated-vs-human-written-text-dataset/AuthentiText_X_2026_AI_vs_Human_Detection_1K.csv')

In [3]:

print("Shape:", df.shape)
print("\nColumn Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
df.head()

Shape: (1000, 12)

Column Types:
 text_id                         object
content_text                    object
author_type                     object
model_source                    object
prompt_complexity_score        float64
perplexity_score               float64
burstiness_index               float64
syntactic_variability          float64
semantic_coherence_score       float64
lexical_diversity_ratio        float64
readability_grade_level        float64
generation_confidence_score    float64
dtype: object

Missing Values:
 text_id                        0
content_text                   0
author_type                    0
model_source                   0
prompt_complexity_score        0
perplexity_score               0
burstiness_index               0
syntactic_variability          0
semantic_coherence_score       0
lexical_diversity_ratio        0
readability_grade_level        0
generation_confidence_score    0
dtype: int64


,text_id,content_text,author_type,model_source,prompt_complexity_score,perplexity_score,burstiness_index,syntactic_variability,semantic_coherence_score,lexical_diversity_ratio,readability_grade_level,generation_confidence_score
0,TXT_0001,learning pattern detection algorithm pattern n...,AI,Human,0.029,73.75,0.953,0.465,0.351,0.187,12.2,0.162
1,TXT_0002,algorithm algorithm data research network mode...,Human,Claude,0.605,43.11,0.054,0.952,0.314,0.636,9.8,0.012
2,TXT_0003,analysis language generation research pattern ...,Human,GPT-4,0.396,59.97,0.709,0.945,0.684,0.500,13.5,0.171
3,TXT_0004,data language system learning content data net...,AI,GPT-4,0.299,18.99,0.532,0.780,0.216,0.103,12.9,0.838
4,TXT_0005,model learning content language model generati...,AI,Human,0.867,82.45,0.478,0.602,0.420,0.198,6.4,0.022


In [4]:
df['char_count'] = df['content_text'].apply(len)
df['word_count'] = df['content_text'].apply(lambda x: len(x.split()))
df['avg_word_length'] = df['char_count'] / (df['word_count'] + 1)

df['sentence_count'] = df['content_text'].apply(lambda x: x.count('.'))
df['avg_sentence_length'] = df['word_count'] / (df['sentence_count'] + 1)

df['unique_words'] = df['content_text'].apply(lambda x: len(set(x.split())))
df['lexical_density'] = df['unique_words'] / (df['word_count'] + 1)

print("Text Features Created!")
df[['char_count','word_count','avg_word_length']].head()

Text Features Created!


,char_count,word_count,avg_word_length
0,120,14,8.000000
1,90,11,7.500000
2,111,12,8.538462
3,78,11,6.500000
4,115,14,7.666667


**"Created linguistic features such as text length, word count, and lexical diversity to capture writing style differences between AI and human text."**

In [5]:
df['uppercase_count'] = df['content_text'].str.count(r'[A-Z]')
df['uppercase_ratio'] = df['uppercase_count'] / (df['char_count'] + 1)

print("Style Features Added!")
df[['uppercase_count','uppercase_ratio']].head()

Style Features Added!


,uppercase_count,uppercase_ratio
0,0,0.0
1,0,0.0
2,0,0.0
3,0,0.0
4,0,0.0


**These features capture stylistic patterns. Human writing often shows irregular punctuation and capitalization compared to AI.**

In [6]:
df['perplexity_burstiness'] = df['perplexity_score'] * df['burstiness_index']
df['coherence_variability'] = df['semantic_coherence_score'] * df['syntactic_variability']
df['readability_lexical'] = df['readability_grade_level'] * df['lexical_diversity_ratio']

print("Interaction Features Created!")
df[['perplexity_burstiness','coherence_variability']].head()

Interaction Features Created!


,perplexity_burstiness,coherence_variability
0,70.28375,0.163215
1,2.32794,0.298928
2,42.51873,0.646380
3,10.10268,0.168480
4,39.41110,0.252840


**Interaction features were created to capture combined effects since individual features showed weak predictive power.**

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=300)
tfidf_matrix = tfidf.fit_transform(df['content_text']).toarray()

tfidf_df = pd.DataFrame(tfidf_matrix, columns=tfidf.get_feature_names_out())

df = pd.concat([df, tfidf_df], axis=1)

print("TF-IDF Applied!")
print("New Shape:", df.shape)

TF-IDF Applied!
New Shape: (1000, 37)


**TF-IDF was used to capture word-level importance, which helps distinguish AI-generated patterns from human writing.**

In [8]:
X = df.drop(['content_text','author_type','text_id','model_source'], axis=1)
y = (df['author_type'] == 'AI').astype(int)

print("Features Shape:", X.shape)
print("Target Distribution:")
print(y.value_counts())

Features Shape: (1000, 33)
Target Distribution:
author_type
0    537
1    463
Name: count, dtype: int64


In [9]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier()
lgb.fit(X, y)

import pandas as pd

lgb_imp = pd.Series(lgb.feature_importances_, index=X.columns).sort_values(ascending=False)

print("Top 15 Important Features:")
lgb_imp.head(15)

[LightGBM] [Info] Number of positive: 463, number of negative: 537
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001733 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5403
[LightGBM] [Info] Number of data points in the train set: 1000, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.463000 -> initscore=-0.148271
[LightGBM] [Info] Start training from score -0.148271
Top 15 Important Features:


prompt_complexity_score        162
generation_confidence_score    146
lexical_diversity_ratio        139
language                       138
perplexity_score               133
perplexity_burstiness          131
semantic_coherence_score       122
network                        120
coherence_variability          117
burstiness_index               115
readability_grade_level        114
syntactic_variability          111
char_count                     108
pattern                        107
readability_lexical            107
dtype: int32

**LightGBM was used for feature importance due to its efficiency and ability to capture non-linear relationships.**

In [10]:
low_features = lgb_imp[lgb_imp < 5].index

print("Features to be removed:", len(low_features))

df.drop(columns=low_features, inplace=True)

print("After Dropping Low Features:")
print(df.shape)

Features to be removed: 4
After Dropping Low Features:
(1000, 33)


**Features with very low importance were removed to reduce noise and improve model performance.**

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier

X = df.drop(['content_text','author_type','text_id','model_source'], axis=1)
y = (df['author_type'] == 'AI').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LGBMClassifier()
model.fit(X_train, y_train)

pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)

print("Model Accuracy After Feature Engineering:", acc)

[LightGBM] [Info] Number of positive: 375, number of negative: 425
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000475 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4877
[LightGBM] [Info] Number of data points in the train set: 800, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.468750 -> initscore=-0.125163
[LightGBM] [Info] Start training from score -0.125163
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

**Model performance improved after feature engineering, indicating that newly created features provide better discriminatory power.**

In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Standardization Done!")
print("Scaled Shape:", X_scaled.shape)

Standardization Done!
Scaled Shape: (1000, 29)


In [13]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

print("Cluster Counts:")
print(df['cluster'].value_counts())

Cluster Counts:
cluster
0    381
2    374
1    245
Name: count, dtype: int64


**K-means clustering was used to create a new feature representing latent group structures in the data.**

In [14]:
df.to_csv("final_feature_engineered.csv", index=False)

print("Final Dataset Saved!")
print("Final Shape:", df.shape)

Final Dataset Saved!
Final Shape: (1000, 34)


# FINAL CONCLUSION
Initial analysis showed that existing features had very low discriminatory power, resulting in poor model performance. Therefore, additional textual, stylistic, and interaction-based features were created. TF-IDF was used to extract word-level patterns. Feature importance analysis using multiple algorithms helped identify relevant features, and less important features were removed. K-means clustering introduced additional structural information. These steps significantly improved the dataset quality for machine learning.